In this notebook, I will model the factor that interconnects the airway resistance between two consecutive days.

In [11]:
import src.models.var_builders as var_builders
import src.data.helpers as dh
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
import src.models.helpers as mh
import src.models.cpts.helpers as cpth

In [12]:
(
    HFEV1,
    ecFEV1,
    AR,
    HO2Sat,
    O2SatFFA,
    IA,
    UO2Sat,
    O2Sat,
    ecFEF2575prctecFEV1,
) = var_builders.o2sat_fev1_fef2575_point_in_time_model_shared_healthy_vars(
    180, 10, "Male"
)

In [13]:
df = dh.load_excel(
    # f"{dh.get_path_to_main()}/ExcelFiles/BR/Refining_F3/infer_AR_with_two_days_model_O2Sat_FEV1.xlsx",
    f"{dh.get_path_to_main()}/ExcelFiles/BR/Refining_F3/infer_AR_with_two_days_model_O2Sat_ecFEV1.xlsx",
    [AR.name],
    ["Day"],
).drop(columns=["Unnamed: 0", HO2Sat.name, IA.name, HFEV1.name])

In [14]:
def get_days_elapsed_df(df_for_ID, n_days_offset=1):
    df_for_ID = df_for_ID.copy()
    # In day 1, put the number of days that has pasat since day 0, repeat for each day

    def get_days_elapsed(curr, prev):
        if prev == None:
            return None
        return (curr - prev).total_seconds() / 3600 / 24

    df_for_ID["AR mean"] = df_for_ID.apply(lambda x: AR.get_mean(x[AR.name]), axis=1)
    # df_for_ID['AR skewness'] = df_for_ID.apply(lambda x: AR.get_skewness(x[AR.name]), axis=1)

    df_for_ID["Prev date"] = df_for_ID.shift(n_days_offset)["Day"]
    df_for_ID["Prev AR mean"] = df_for_ID.shift(n_days_offset)["AR mean"]
    # df_for_ID['Prev AR skewness'] = df_for_ID.shift(n_days_offset)['AR skewness']

    df_for_ID["Days elapsed"] = df_for_ID.apply(
        lambda x: get_days_elapsed(x["Day"], x["Prev date"]), axis=1
    )
    df_for_ID["AR mean shift"] = df_for_ID["AR mean"] - df_for_ID["Prev AR mean"]
    # df_for_ID['AR skewness shift'] = df_for_ID['AR skewness'] - df_for_ID['Prev AR skewness']

    return df_for_ID[["ID", "Day", "Days elapsed", "AR mean shift"]]
    # return df_for_ID[['ID', 'Day', 'Days elapsed', 'AR mean shift', 'AR skewness shift']]


# out = df.groupby('ID').apply(get_days_elapsed_df).reset_index(drop=True)

## Compute day elapsed between two consecutive entries

In [5]:
df1 = df.merge(
    df.groupby("ID").apply(get_days_elapsed_df).reset_index(drop=True),
    on=["ID", "Day"],
    how="inner",
)

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_51455/865564159.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df.groupby("ID").apply(get_days_elapsed_df).reset_index(drop=True),


In [74]:
df1.head()

,ID,Day,Airway resistance (%),Days elapsed,AR mean shift
0,101,2019-01-25,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",NaN,NaN
1,101,2019-01-26,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,-1.341903
2,101,2019-01-27,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,2.332655
3,101,2019-01-28,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,0.000000
4,101,2019-01-29,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,-0.901317


### Validate the output

In [11]:
df1.describe()

,Days passed,AR mean shift
count,40908.000000,40908.000000
mean,4.737020,-0.036692
std,20.357683,4.347104
min,1.000000,-45.480774
25%,1.000000,-1.634580
50%,1.000000,0.000000
75%,3.000000,1.607764
max,980.000000,53.672746


In [12]:
df1[df1["Days elapsed"] > 100]

,ID,Day,Airway resistance (%),Days passed,AR mean shift
2296,103,2023-09-25,"[0.0, 0.0, 0.0, 0.0, 6.27525413e-05, 0.0003433...",192.0,2.478966
2406,104,2020-03-23,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",152.0,-8.661761
2476,104,2023-03-20,"[0.0214458477, 0.0266964061, 0.0329117122, 0.0...",708.0,-5.112841
2787,106,2021-11-11,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",117.0,0.179750
2858,106,2023-01-18,"[0.0, 0.0, 0.0, 0.0, 1.09412661e-05, 0.0001043...",146.0,6.280519
...,...,...,...,...,...
38909,507,2022-08-09,"[0.06523372, 0.08521032, 0.09294863, 0.0988228...",125.0,-0.430656
39702,513,2023-05-26,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",308.0,33.354319
39784,514,2023-10-17,"[0.0, 0.000309536674, 0.0019575308, 0.00472002...",110.0,-0.579326
40141,518,2022-09-16,"[0.0, 9.20574027e-05, 0.00250103785, 0.0067597...",117.0,2.459672


In [13]:
# Verify that the prev day is indeed correct)
df1.iloc[2295:2297]
# Count number of None
print(df1["Days elapsed"].isna().sum())
# Count number if ids
print(df1["ID"].nunique())
# They should be equal

352
352


In [14]:
df1[df1["AR mean shift"] > 20]
df1.iloc[2537:2539]

,ID,Day,Airway resistance (%),Days passed,AR mean shift
2537,104,2023-10-31,"[0.00645966881, 0.00855669673, 0.011294808, 0....",6.0,-3.175633
2538,104,2023-11-06,"[0.0161213898, 0.0203813231, 0.0255672073, 0.0...",6.0,-4.565804


### Analyse time between two consecutive entries

In [16]:
vc = df1["Days elapsed"].value_counts()
# 1/3 of the consecutive indices are more than 1 day apart (~10k entries)
# 97% of the entries are less than 5 days apart from the previous entry
# For the CPT, I'll take 1, 2, 3, 4, 5 days apart, then avg 6-50 -> this last up to the max days diff


# Plot the histogram with vc index and vc values
fig = px.bar(x=vc.index, y=vc.values / sum(vc.values) * 100)
# Set x axis label to day to day difference
fig.update_xaxes(
    title_text="Number of days between two consecutive entries",
    range=[0, 30],
    tickvals=list(range(0, 31, 1)),
)
# Set y axis label to percentage
fig.update_yaxes(
    title_text="Percentage of total entries (%)", tickvals=[2] + list(range(0, 55, 5))
)

title = "Distribution of the time between two measurements"
# Set title
fig.update_layout(title=title, width=800, height=350, font=dict(size=10))

fig.show()

# Save figure
fig.write_image(
    f"{dh.get_path_to_main()}/PlotsBreathe/Interconnecting_ARs_entries/{title}.pdf"
)

In [123]:
# Study per ID
# Get idx at which the days elapsed is more than 3

df1[df1["Days elapsed"] > 3].index

def get_idx_more_than_n_days_elapsed(df, n=3):
    df = df.reset_index()
    n_days_total = df.shape[0]
    df_tmp = df[df["Days elapsed"] > n]
    if df_tmp.empty:
        return n_days_total, n_days_total
    n_days_consec = df_tmp.index[0]
    return n_days_consec, n_days_total

s_n_entries_to_break = df1.groupby('ID').apply(lambda x: get_idx_more_than_n_days_elapsed(x, 3)).sort_values(ascending=False)

s_n_entries_to_break

ID
405    (1035, 1035)
101     (592, 1680)
272      (418, 800)
201      (290, 509)
203      (286, 845)
           ...     
213          (1, 1)
225          (1, 1)
354          (1, 1)
516          (1, 1)
355          (1, 1)
Length: 352, dtype: object

In [133]:
df1[df1.ID == '101']

,ID,Day,Airway resistance (%),Days elapsed,AR mean shift
0,101,2019-01-25,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",NaN,NaN
1,101,2019-01-26,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,-1.341903
2,101,2019-01-27,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,2.332655
3,101,2019-01-28,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,0.000000
4,101,2019-01-29,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",1.0,-0.901317
...,...,...,...,...,...
1675,101,2023-11-08,"[0.000638900028, 0.0011020269, 0.00141903779, ...",1.0,-1.550963
1676,101,2023-11-09,"[0.0, 5.38513047e-05, 0.000369937887, 0.000911...",1.0,3.158726
1677,101,2023-11-10,"[0.000149953968, 0.000505965182, 0.00103667253...",1.0,-1.607764
1678,101,2023-11-11,"[0.000149953968, 0.000505965182, 0.00103667253...",1.0,0.000000


## Compute shift in AR mean

In [15]:
# Build aggregate df of shift in AR for different offsets

df_mixed_offset = pd.DataFrame()

max_days_elapsed = 1

for n_days_offset in range(1, max_days_elapsed + 1):
    df_offset = (
        df.groupby("ID")
        .apply(lambda row: get_days_elapsed_df(row, n_days_offset))
        .reset_index(drop=True)
    )
    df_offset["Offset"] = n_days_offset
    # Remove nan
    df_offset = df_offset.dropna()

    # Add to mix offset
    df_mixed_offset = pd.concat([df_mixed_offset, df_offset])

/var/folders/zq/v2r6yn111s3gpdf8lzf72xvw0000gn/T/ipykernel_51455/1314306724.py:10: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



### Study the shift in AR mean

In [16]:
# Scatter plot with days elapsed on x axis and AR diff on y axis, using px
y_col = "AR mean shift"
# y_col = 'AR skewness shift'
fig = px.scatter(df_mixed_offset, x="Days elapsed", y=y_col, color="ID")
# Set x axis range to 0-100
fig.update_xaxes(range=[0, 200])
fig.update_xaxes(range=[0, 50], title="Number of days elapsed")
# Add more y axi tick vals
fig.update_yaxes(title="Mean airway resistance shift (%)")
# Reduce marker size
fig.update_traces(marker=dict(size=2))
title = f"How much does the airway resistance change between different time periods (max {max_days_elapsed} days elapsed)?"
# title = f"How much does the airway resistance change between different time periods (max {max_days_elapsed} days elapsed)<br>(AR inferred with non smoothed FEV1)?"
fig.update_layout(
    title=title, width=800, height=400, font=dict(size=10), showlegend=False
)
fig.show()
fig.write_image(
    f"{dh.get_path_to_main()}/PlotsBreathe/Interconnecting_ARs_entries/{title}.pdf"
)

In [6]:
# I want to see the distribution of AR diffs for each day elapsed
from scipy.stats import norm

y_col = "AR mean shift"

fig = make_subplots(rows=6, cols=1, shared_xaxes=True)
# xbin_size = 0.2
xbin_size = 1
# xbin_absolute_span = 50
xbin_absolute_span = 10
xbins = dict(
    start=-xbin_absolute_span - 0.5, end=xbin_absolute_span + 0.5, size=xbin_size
)


def add_plot_for_offset(offset, row):
    df_tmp = df_mixed_offset[df_mixed_offset["Days elapsed"] == offset]
    print(offset, df_tmp.shape)
    fig.add_trace(
        go.Histogram(
            x=df_tmp[y_col],
            xbins=xbins,
            histnorm="probability",
            name=f"{offset} days offset",
        ),
        row=row,
        col=1,
    )
    # Model the data by a normal distribution
    mean = df_tmp[y_col].mean()
    std = df_tmp[y_col].std()
    x = list(range(-10, 11))
    y = norm.pdf(x, loc=mean, scale=std)
    # Add trace
    # fig.add_trace(go.Scatter(x=x, y=y, mode='lines', name=f"Normal distribution for {offset} days offset"), row=row, col=1)


# for offset in range(1, 51):
#     add_plot_for_offset(offset, offset)

add_plot_for_offset(1, 1)
add_plot_for_offset(2, 2)
add_plot_for_offset(8, 3)
add_plot_for_offset(14, 4)
add_plot_for_offset(20, 5)
add_plot_for_offset(50, 6)

# Set y axis range to 0, 0.6
fig.update_yaxes(range=[0, 0.58])
# Set x axis label
fig.update_xaxes(title_text="Shift in mean airway resistance (%)", row=6, col=1)
# fig.update_xaxes(title_text='Change in skewness of airway resistance (%)', row=6, col=1)
# Add x axis tick vals
fig.update_xaxes(tickvals=np.arange(-10, 11, 1), row=6, col=1)
# fig.update_xaxes(tickvals=np.arange(-50, 55, 5), row=6, col=1)
# Update layout
title = f"Shift in airway resistance for different time periods elapsed (bin_width = {xbin_size}%, bin_span = {xbin_absolute_span}, raw FEV1)"
# fig.update_layout(height=2600, width=1000, title=title)
fig.update_layout(height=600, width=1000, title=title)
# Save image
# fig.write_image(f"{dh.get_path_to_main()}/PlotsBreathe/Interconnecting_ARs_entries/{title}.pdf")
fig.show()

1 (21855, 5)
2 (22155, 5)
8 (2988, 5)
14 (1773, 5)
20 (556, 5)
50 (84, 5)


### Build CPT assuming only the 1st moment changes (not skewness or spread)


In [6]:
# Building P(AR_next | days_elapsed, AR_prev)
# import src.models.helpers as mh
import numpy as np
import src.modelling_ar.ar as model_ar

AR1 = mh.VariableNode(
    "Airway resistance day 1 (%)", 0, 90, 2, prior={"type": "uniform"}
)
AR2 = mh.VariableNode(
    "Airway resistance day 2 (%)", 0, 90, 2, prior={"type": "uniform"}
)
DE = mh.DiscreteVariableNode("Days elapsed", 1, max_days_elapsed, 1)

In [7]:
def calc_cpt(
    AR_next_day: mh.VariableNode,
    AR_curr_day: mh.VariableNode,
    DE: mh.DiscreteVariableNode,
    shift_p,
    shift_val,
    tol=1e-6,
    debug=False,
):
    cpt = np.empty([AR_next_day.card, AR_curr_day.card, DE.card])

    for i, de in enumerate(DE.values):
        # For each shift value, get the mapping AR -> AR_next_day for each shifted bin in AR
        # Weight the result by the probability of that shift
        # Add it to the CPT for this day
        for s in range(len(shift_val)):
            if debug:
                print(f"Computing CPT for days elapsed={de}, shift={shift_val[s]}")
            # Summing over the columns of the cpt returned by calc_cpt_X_plus_k should give 1, except at the boundaries
            # Since we weight the 1s by a probability of shift that also sums to one, the sum of the cpt should be 1 (except at the boundaries, see below)
            cpt[:, :, i] += shift_p[i, s] * calc_cpt_X_plus_k(
                AR_curr_day,
                AR_next_day,
                shift_val[s],
                tol=tol,
                debug=debug,
            )
        # Normalise the CPT along axis 0 (AR_next_day)
        total = np.sum(cpt[:, :, i], axis=0)
        print(f"Sum along axis 0 before normalisation: np.sum(cpt[:, :, {i}], axis=0) = {total}")
        cpt[:, :, i] /= total

        # Check that the sum of probabilities is 1
        total = np.sum(cpt[:, :, i], axis=0)
        assert (
            (abs(total - 1) < tol).all()
        ), f"The sum of the probabilities should be 1, got sum(cpt)={total}])"
    return cpt


def calc_cpt_X_plus_k(
    Z: mh.VariableNode,
    X: mh.VariableNode,
    k,
    tol=1e-6,
    debug=False,
):
    """
    Computes the CPT for P(Z|X, Y), when Z is shifted from X by a constant value k
    Z = X + k
    X: parent variable
    Z: child variable
    k: constant, positive or negative

    We compute the CPT with a shift and conquer method:
    1) Start with a CPT zeroed out probabilities
    2) Shift all X bin intervals by the drop amount
    3) For each shifted X bin, spread the X bin evenly onto the overlapping Z bins
    4) Normalise the CPT

    This allows the function to be agnostic of how X and Z are binned.

    - What happens when the function is shifted outside the boundary? -> Raise an error as it shouldn't happen by how the model is built
    """
    nbinsX = len(X.bins)
    nbinsZ = len(Z.bins)

    cpt = np.zeros([nbinsZ, nbinsX])

    for i in range(nbinsX):
        shifted_X_bin_low = X.bins[i] + k
        shifted_X_bin_up = (X.bins[i] + X.bin_width) + k
        if debug:
            print(
                f"Shifting X bin {i} from [{X.bins[i]};{X.bins[i]+X.bin_width}) to [{shifted_X_bin_low};{shifted_X_bin_up}), shift amount={k}%"
            )
        # If the shifted bin is outside the boundaries of Z, continue:
        if shifted_X_bin_low >= (Z.bins[-1] + Z.bin_width) or shifted_X_bin_up <= Z.bins[0]:
            if debug:
                print(f"Shift outside boundaries of Z.bins=[{Z.bins[0]};{Z.bins[-1] + Z.bin_width})")
            continue
        # Handle the case where the shifted bin is partially outside the boundaries
        # Adjust the boundaries of the shifted bin to be within the boundaries of Z
        if shifted_X_bin_low < Z.bins[0]:
            if debug:
                print("Shift partially outside boundaries, adjusting lower boundary")
            shifted_X_bin_low = Z.bins[0]
        if shifted_X_bin_up > Z.bins[-1] + Z.bin_width:
            if debug:
                print("Shift partially outside boundaries, adjusting upper boundary")
            shifted_X_bin_up = Z.bins[-1] + Z.bin_width

        bin_contribution = mh.get_bin_contribution_to_cpt(
            [shifted_X_bin_low, shifted_X_bin_up], Z.bins, debug=debug
        )
        if debug:
            print(f"i={i}/{nbinsX-1}, z={bin_contribution}")
        # There is just one bin contribution to the CPT
        cpt[:, i] = bin_contribution

    # total = np.sum(cpt)
    # if debug:
    #     print(f"Results before normalisation sum(cpt)={total}")
    # cpt /= total

    # Raise if sum of probabilities is not 1
    # total = np.sum(cpt, axis=0)
    # assert (
    #     (abs(total - 1) < tol).all()
    # ), f"The sum of the probabilities should be 1, got sum(cpt)={total}])"

    return cpt

In [8]:
# Build the shift distributions
shift_val = np.arange(-5, 6, 1)
shift_p = np.empty((max_days_elapsed, len(shift_val)))
for i, de in enumerate(DE.values):
    print("days elapsed: ", de)
    mean_shift = df_mixed_offset[df_mixed_offset["Days elapsed"] == de]["AR mean shift"]
    # Bin up the mean shift series into bins starting at -5 and ending at 5, with bin size 1
    shift_p[i, :] = np.histogram(
        mean_shift, bins=np.arange(-5.5, 6.5, 1), density=True
    )[0]

print("shift probability shape: ", shift_p.shape)
print("shift_val: ", shift_val)

days elapsed:  1
days elapsed:  2
days elapsed:  3
shift probability shape:  (3, 11)
shift_val:  [-5 -4 -3 -2 -1  0  1  2  3  4  5]


In [9]:
cpt = calc_cpt(AR2, AR1, DE, shift_p, shift_val, debug=True)

Computing CPT for days elapsed=1, shift=-5
Shifting X bin 0 from [0.0;2.0) to [-5.0;-3.0), shift amount=-5%
Shift outside boundaries of Z.bins=[0.0;90.0)
Shifting X bin 1 from [2.0;4.0) to [-3.0;-1.0), shift amount=-5%
Shift outside boundaries of Z.bins=[0.0;90.0)
Shifting X bin 2 from [4.0;6.0) to [-1.0;1.0), shift amount=-5%
Shift partially outside boundaries, adjusting lower boundary
k=0, bin=[0.0;2.0], p=1.0 (=1.0/1.0)
i=2/44, z=[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Shifting X bin 3 from [6.0;8.0) to [1.0;3.0), shift amount=-5%
k=0, bin=[0.0;2.0], p=0.5 (=1.0/2.0)
k=1, bin=[2.0;4.0], p=0.5 (=1.0/2.0)
i=3/44, z=[0.5 0.5 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
 0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
 0.  0.  0.  0.  0.  0.  0.  0.  0. ]
Shifting X bin 4 from [8.0;10.0) to [3.0;5.0), shift amount=-5%
k=1, bin=[2.0;4.0], p=0.5 (=1.0/

#### Plot CPT relationships

In [12]:
import src.inference.helpers as ih
def compare_ARs_for_one_entry(idx):
    title = f"P(AR_next | AR_prev, days_elapsed) for diffent days elapsed (idx {idx})"
    fig = make_subplots(rows=1, cols=1, shared_xaxes=True)
    ih.plot_histogram(fig, AR1, df.loc[idx, AR.name], AR1.a, AR1.b, 1, 1, name="AR day 1", annot=False)
    AR_next_day_p = np.matmul(cpt[:, :, 0], df.loc[idx, AR.name])
    ih.plot_histogram(fig, AR2, AR_next_day_p, AR2.a, AR2.b, 1, 1, name="AR day 2, days elapsed=1", annot=False)
    AR_next_day_p = np.matmul(cpt[:, :, 2], df.loc[idx, AR.name])
    ih.plot_histogram(fig, AR2, AR_next_day_p, AR2.a, AR2.b, 1, 1, name="AR day 2, days elapsed=3", annot=False)
    # Add x axis title
    fig.update_xaxes(title_text="Airway resistance (%)", row=1, col=1)
    # Reduce figure height
    fig.update_layout(height=200, width=1000, title=title, font=dict(size=10))
    # remove marings
    fig.update_layout(margin=dict(l=2, r=2, t=30, b=2))
    fig.show() 
    # Save figure
    fig.write_image(f"{dh.get_path_to_main()}/PlotsBreathe/Interconnecting_ARs_entries/{title}.pdf")

compare_ARs_for_one_entry(20000)
# compare_ARs_for_one_entry(21000)
compare_ARs_for_one_entry(1000)
compare_ARs_for_one_entry(4400)

In [14]:
de = 3
fig, title = cpth.plot_2d_cpt(cpt[:, :, de - 1], AR2, AR1, 3000, invert=False)
# Update font
title =  title + f", {de} days elapsed, shift span [-5;5]"
fig.update_layout(font=dict(size=7), title = title)
fig.show()

# Save figure
fig.write_image(
    f"{dh.get_path_to_main()}/PlotsBreathe/Interconnecting_ARs_entries/{title}.pdf"
)

#### Save CPT

In [13]:
# Save cpt
cpth.save_cpt([AR2, AR1, DE], cpt, suffix=f"_shift_span_[-5;5]")

## Study the shift per bin

In [14]:
df

,ID,Day,Airway resistance (%),AR mean
0,101,2019-01-25,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",55.623870
1,101,2019-01-26,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",54.281967
2,101,2019-01-27,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",56.614623
3,101,2019-01-28,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",56.614623
4,101,2019-01-29,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",54.281967
...,...,...,...,...
41255,553,2023-10-08,"[0.0, 0.03160499, 0.10498536, 0.11549848, 0.11...",12.840323
41256,553,2023-10-11,"[0.0, 0.03160499, 0.10498536, 0.11549848, 0.11...",12.840323
41257,553,2023-11-06,"[0.0, 0.03160499, 0.10498536, 0.11549848, 0.11...",12.840323
41258,553,2023-11-08,"[0.0, 0.03160499, 0.10498536, 0.11549848, 0.11...",12.840323


In [218]:
AR.midbins

array([ 1.,  3.,  5.,  7.,  9., 11., 13., 15., 17., 19., 21., 23., 25.,
       27., 29., 31., 33., 35., 37., 39., 41., 43., 45., 47., 49., 51.,
       53., 55., 57., 59., 61., 63., 65., 67., 69., 71., 73., 75., 77.,
       79., 81., 83., 85., 87., 89.])

In [42]:
df_exploded = df1.copy()

for i, row in df_exploded[0:10].iterrows():
    row = pd.DataFrame(data=row[AR.name])
    df_exploded = pd.concat([df_exploded, row], axis=1)

df_exploded

,ID,Day,Airway resistance (%),AR mean,Days passed,AR diff,0,0,0,0,0,0,0,0,0,0
0,101,2019-01-25,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",55.623870,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,101,2019-01-26,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",54.281967,1.0,-1.341903,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,101,2019-01-27,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",56.614623,1.0,2.332655,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,101,2019-01-28,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",56.614623,1.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,101,2019-01-29,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",54.281967,1.0,-2.332655,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41255,553,2023-10-08,"[0.0, 0.03160499, 0.10498536, 0.11549848, 0.11...",12.840323,2.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41256,553,2023-10-11,"[0.0, 0.03160499, 0.10498536, 0.11549848, 0.11...",12.840323,3.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41257,553,2023-11-06,"[0.0, 0.03160499, 0.10498536, 0.11549848, 0.11...",12.840323,26.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
41258,553,2023-11-08,"[0.0, 0.03160499, 0.10498536, 0.11549848, 0.11...",12.840323,2.0,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
